# AME end-to-end example (genriesz)

This notebook demonstrates how to estimate an **Average Marginal Effect (AME)**,
i.e., an **average derivative** of the outcome regression function.

We simulate

\[
Y = \sin(X_0) + 0.5 X_1^2 + \varepsilon,
\]

so the true AME for coordinate 0 is

\[
\mathbb{E}[\partial_{x_0} \gamma(X)] = \mathbb{E}[\cos(X_0)].
\]

If \(X_0 \sim N(0,1)\), then \(\mathbb{E}[\cos(X_0)] = \exp(-1/2)\).


In [ ]:
import numpy as np
from genriesz import grr_ame, SquaredGenerator, PolynomialBasis

rng = np.random.default_rng(0)


## Synthetic data with known true AME

In [ ]:
n = 4000
d = 3

X = rng.normal(size=(n, d))
eps = rng.normal(scale=1.0, size=n)

Y = np.sin(X[:, 0]) + 0.5 * (X[:, 1] ** 2) + eps

true_ame0 = float(np.exp(-0.5))  # E[cos(N(0,1))]
print("Approx. true AME for coordinate 0:", true_ame0)


## Fit GRR for AME

In [ ]:
# A simple polynomial basis on X
basis = PolynomialBasis(degree=3, include_bias=True)

gen = SquaredGenerator(C=0.0).as_generator()

res = grr_ame(
    X=X,
    Y=Y,
    coordinate=0,
    basis=basis,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res.summary_text())
